## Find frequent item sets through association rule mining

In [ ]:
from itertools import combinations
from collections import defaultdict


dataset = {
    'T1': {'Bread', 'Milk'},
    'T2': {'Bread', 'Diaper', 'Beer', 'Eggs'},
    'T3': {'Diaper', 'Beer', 'Cake', 'Milk'},
    'T4': {'Bread', 'Milk', 'Diaper', 'Beer'},
    'T5': {'Bread', 'Milk', 'Diaper', 'Cake'}
}

transactions = list(dataset.values())

MIN_SUPPORT = 0.6 # 60%
MIN_CONFIDENCE = 0.7 # Added for rule generation

def get_frequent_itemsets(transactions, min_support):
    num_transactions = len(transactions)
    frequent_itemsets = {}
    current_L_k = {}
    item_counts = defaultdict(int)
    for transaction in transactions:
        for item in transaction:
            item_counts[frozenset([item])] += 1

    current_L_k = {
        itemset: count / num_transactions
        for itemset, count in item_counts.items()
        if count / num_transactions >= min_support
    }
    frequent_itemsets.update(current_L_k)

    k = 2
    while current_L_k: # Loop until no more frequent itemsets are found

        all_prev_frequent_items = set()
        for itemset in current_L_k:
            all_prev_frequent_items.update(itemset)

        candidates_Ck = set()
        for combo in combinations(all_prev_frequent_items, k):
            candidates_Ck.add(frozenset(combo))

        # Count support for candidates
        candidate_counts = defaultdict(int)
        for transaction in transactions:
            for candidate in candidates_Ck:
                if candidate.issubset(transaction):
                    candidate_counts[candidate] += 1


        current_L_k = {
            candidate: count / num_transactions
            for candidate, count in candidate_counts.items()
            if count / num_transactions >= min_support
        }
        frequent_itemsets.update(current_L_k)
        k += 1

    return frequent_itemsets

def generate_association_rules(frequent_itemsets, min_confidence):
    rules = []
    for itemset, support in frequent_itemsets.items():
        if len(itemset) > 1:
            for i in range(1, len(itemset)):
                for antecedent in combinations(itemset, i):
                    antecedent = frozenset(antecedent)
                    consequent = itemset - antecedent
                    antecedent_support = frequent_itemsets[antecedent]

                    confidence = support / antecedent_support
                    if confidence >= min_confidence:
                        rules.append((antecedent, consequent, confidence, support))
    return rules


# 1. Get Frequent Itemsets
freq_items = get_frequent_itemsets(transactions, MIN_SUPPORT)

print("--- FREQUENT ITEMSETS (Min Support: 60%) ---")
print(f"{'Itemset':<30} | {'Support':<10}")
print("-" * 45)
for itemset, support in sorted(freq_items.items(), key=lambda x: (len(x[0]), -x[1])):
    item_names = ", ".join(list(itemset))
    print(f"{'[' + item_names + ']':<30} | {support:.2f} ({int(support * len(transactions))}/{len(transactions)})")

# 2. Generate Association Rules
rules = generate_association_rules(freq_items, MIN_CONFIDENCE)

print("\n--- ASSOCIATION RULES (Min Confidence: 70%) ---")
print(f"{'Rule':<30} | {'Confidence':<10} | {'Support'}")
print("-" * 60)
for antecedent, consequent, confidence, support in sorted(rules, key=lambda x: -x[2]):
    rule_str = f"[{', '.join(antecedent)}] => [{', '.join(consequent)}]"
    print(f"{rule_str:<30} | {confidence:.2f}       | {support:.2f}")

--- FREQUENT ITEMSETS (Min Support: 60%) ---
Itemset                        | Support   
---------------------------------------------
[Milk]                         | 0.80 (4/5)
[Bread]                        | 0.80 (4/5)
[Diaper]                       | 0.80 (4/5)
[Beer]                         | 0.60 (3/5)
[Milk, Bread]                  | 0.60 (3/5)
[Diaper, Beer]                 | 0.60 (3/5)
[Bread, Diaper]                | 0.60 (3/5)
[Milk, Diaper]                 | 0.60 (3/5)

--- ASSOCIATION RULES (Min Confidence: 70%) ---
Rule                           | Confidence | Support
------------------------------------------------------------
[Beer] => [Diaper]             | 1.00       | 0.60
[Milk] => [Bread]              | 0.75       | 0.60
[Bread] => [Milk]              | 0.75       | 0.60
[Diaper] => [Beer]             | 0.75       | 0.60
[Bread] => [Diaper]            | 0.75       | 0.60
[Diaper] => [Bread]            | 0.75       | 0.60
[Milk] => [Diaper]             | 0.75      

one hundred people were interviwed outside a chocolate shop to find out which flavour of chocolate cream they preferred. the results are given in the table beloow classified by gender.

        Strawberry|coffee|orange|vanilla|
male   |23        |18    |08    |08     |
female |15        |06    |12    |10     |


we want to determine wheteher the flavour of chocolate cream is indepenedent of gender
a. state the null hypothesis and the alternate hypothesis.
b. calculate the degree of freedom
c.
create the expected frequency table for this data.

In [10]:
import numpy as np
import pandas as pd

# 1. Observed Data Setup
observed_df = pd.DataFrame(
    [[23, 18, 8, 8], [15, 6, 12, 10]],
    index=['Male', 'Female'],
    columns=['Strawberry', 'Coffee', 'Orange', 'Vanilla'],
)

# --- Part a: Hypotheses ---
print('=== a. Hypotheses ===')
print(
    'H0 (Null Hypothesis)     : Chocolate cream flavor preference is'
    ' independent of gender.'
)
print(
    'H1 (Alternate Hypothesis): Chocolate cream flavor preference is dependent'
    ' on gender.\n'
)

# --- Part b: Degrees of Freedom ---
rows, cols = observed_df.shape
dof = (rows - 1) * (cols - 1)

print('=== b. Degrees of Freedom ===')
print(f'Formula: (Rows - 1) * (Columns - 1)')
print(f'df = ({rows} - 1) * ({cols} - 1) = {dof}\n')

# --- Part c: Expected Frequency Table ---
row_totals = observed_df.sum(axis=1)
col_totals = observed_df.sum(axis=0)
grand_total = observed_df.values.sum()

# E = (Row Total * Column Total) / Grand Total
expected_matrix = np.outer(row_totals, col_totals) / grand_total

# Create DataFrame and add margin totals
expected_df = pd.DataFrame(
    expected_matrix, index=observed_df.index, columns=observed_df.columns
)
expected_df['Total'] = row_totals
expected_df.loc['Total'] = list(col_totals) + [grand_total]

print('=== c. Expected Frequency Table ===')
print(expected_df.round(2))

=== a. Hypotheses ===
H0 (Null Hypothesis)     : Chocolate cream flavor preference is independent of gender.
H1 (Alternate Hypothesis): Chocolate cream flavor preference is dependent on gender.

=== b. Degrees of Freedom ===
Formula: (Rows - 1) * (Columns - 1)
df = (2 - 1) * (4 - 1) = 3

=== c. Expected Frequency Table ===
        Strawberry  Coffee  Orange  Vanilla  Total
Male         21.66   13.68    11.4    10.26     57
Female       16.34   10.32     8.6     7.74     43
Total        38.00   24.00    20.0    18.00    100
